In [29]:
import os

import numpy as np
import pandas as pd
import scipy.stats
from sklearn.metrics import cohen_kappa_score

In [30]:
# -------------------------------------------------------------------
# Configuration
# -------------------------------------------------------------------

target_cols = ["omim_id"]
rater_col = "username"
present_col = "present_hpo_terms"
absent_col = "absent_hpo_terms"
include_unknown = False
confidence = 0.95

csv_path = os.path.join('data', "Face2HPO-181120251320.csv")
df = pd.read_csv(csv_path)
df

,image_id,user_id,username,patient_id,omim_id,present_hpo_terms,present_hpo_terms_readable,absent_hpo_terms,absent_hpo_terms_readable
0,7345,2,petkraw2,4978,147920 PS147920,HP:0000160 HP:0000233 HP:0000219 HP:0000303 HP...,Narrow mouth; Thin vermilion border; Thin uppe...,HP:0410030 HP:0000154 HP:0012471 HP:0000179 HP...,Cleft lip; Wide mouth; Thick vermilion border;...
1,14473,2,petkraw2,9117,105830,HP:0000307 HP:0000446 HP:0002000 HP:0000322,Pointed chin; Narrow nasal bridge; Short colum...,HP:0410030 HP:0000232 HP:0010803 HP:0000347 HP...,Cleft lip; Everted lower lip vermilion; Everte...
2,22785,261,stefanbarakat,12980,PS135900,HP:0410030 HP:0000154 HP:0002714 HP:0012471 HP...,Cleft lip; Wide mouth; Downturned corners of m...,HP:0000160 HP:0010805 HP:0000233 HP:0000219 HP...,Narrow mouth; Upturned corners of mouth; Thin ...
3,19723,2,petkraw2,11619,147920 PS147920,HP:0000233 HP:0000219 HP:0000431 HP:0003196 HP...,Thin vermilion border; Thin upper lip vermilio...,HP:0410030 HP:0012471 HP:0000179 HP:0010803 HP...,Cleft lip; Thick vermilion border; Thick lower...
4,8863,940,leejiani,5864,615866 PS135900,HP:0000154 HP:0002714 HP:0000233 HP:0000219 HP...,Wide mouth; Downturned corners of mouth; Thin ...,HP:0000160 HP:0010805 HP:0012471 HP:0000179 HP...,Narrow mouth; Upturned corners of mouth; Thick...
...,...,...,...,...,...,...,...,...,...
1671,18504,2562,GenesMD,10980,147920,HP:0000160 HP:0000233 HP:0000219 HP:0000430 HP...,Narrow mouth; Thin vermilion border; Thin uppe...,HP:0410030 HP:0000154 HP:0012471 HP:0000179 HP...,Cleft lip; Wide mouth; Thick vermilion border;...
1672,12070,2562,GenesMD,7562,300855,HP:0000154 HP:0012471 HP:0000179 HP:0000232 HP...,Wide mouth; Thick vermilion border; Thick lowe...,HP:0410030 HP:0000160 HP:0000233 HP:0000219 HP...,Cleft lip; Narrow mouth; Thin vermilion border...
1673,19642,2594,nakmullin,11576,PS163950,HP:0000154 HP:0002714 HP:0000307,Wide mouth; Downturned corners of mouth; Point...,HP:0410030 HP:0000160 HP:0010805,Cleft lip; Narrow mouth; Upturned corners of m...
1674,19237,2223,Emilia_Athanasiou,11386,PS163950,HP:0410030,Cleft lip,NaN,NaN


In [31]:
# -------------------------------------------------------------------
# Helpers
# ------------------------------------------------------------------

def split_terms(s):
    if pd.isna(s) or not isinstance(s, str):
        return []
    s = s.strip()
    if not s:
        return []
    return s.split()


def pairwise_kappa_for_target(hpo_df_target: pd.DataFrame) -> pd.DataFrame:
    """
    Compute pairwise Cohen's kappa between raters for a single target.
    Ratings are taken over all HPOs for that target.
    """
    # Build wide HPO × rater matrix
    wide = hpo_df_target.pivot(index="hpo", columns="rater", values="rating")

    raters = wide.columns.tolist()
    results = []

    for i in range(len(raters)):
        for j in range(i + 1, len(raters)):
            r1, r2 = raters[i], raters[j]
            # Drop HPOs where both raters have -1 (no annotation from either)
            sub = wide[[r1, r2]].dropna()
            mask = ~((sub[r1] == -1) & (sub[r2] == -1))
            sub = sub[mask]

            if sub.shape[0] == 0:
                continue

            kappa = cohen_kappa_score(sub[r1], sub[r2])
            results.append(
                {"rater_1": r1, "rater_2": r2, "kappa": kappa}
            )

    return pd.DataFrame(results)


# -------------------------------------------------------------------
# Fleiss' kappa (standard definition) for a table of items × categories
# -------------------------------------------------------------------
def fleiss_kappa(counts):
    """
    counts_matrix: shape (N, k)
    N = number of items (here: image_id–HPO combinations for a given HPO)
    k = number of categories (here: 3: unknown / absent / present)
    Each row i: counts n_ij of raters assigning item i to category j.

    Returns standard Fleiss' kappa across the N items. [web:36]
    """
    # Total number of ratings per item (may vary)
    n_i = counts.sum(axis=1)  # shape (N,)

    # Items that got fewer than 2 ratings cannot contribute to agreement.
    valid = n_i > 1
    if not valid.any():
        return np.nan

    counts_valid = counts[valid]
    n_i_valid = n_i[valid]

    # Observed agreement per item P_i
    P_i = (np.sum(counts_valid ** 2, axis=1) - n_i_valid) / (n_i_valid * (n_i_valid - 1))

    # Overall observed agreement
    P_bar = P_i.mean()

    # Category proportions across all valid ratings
    N_total_ratings = n_i_valid.sum()
    p_j = counts_valid.sum(axis=0) / N_total_ratings

    # Expected agreement
    P_e = np.sum(p_j ** 2)

    if P_e == 1:
        return 1.0

    kappa = (P_bar - P_e) / (1.0 - P_e)
    return kappa

In [32]:
# -------------------------------------------------------------------
# Build long-format ratings: one row per (image_id, hpo, rater)
# -------------------------------------------------------------------
records = []

for (image_id), group in df.groupby("image_id"):
    raters = group[rater_col].unique().tolist()

    # For each rater, collect present/absent terms on this image
    per_rater_present = {}
    per_rater_absent = {}
    all_hpos = set()

    for _, row in group.iterrows():
        r = row[rater_col]
        present_terms = set(split_terms(row.get(present_col, "")))
        absent_terms = set(split_terms(row.get(absent_col, "")))
        per_rater_present[r] = present_terms
        per_rater_absent[r] = absent_terms
        all_hpos.update(present_terms)
        all_hpos.update(absent_terms)

    # Now define items: each HPO term on this image
    # For each item, every rater gives a rating in {0,1,2}
    for hpo in sorted(all_hpos):
        for r in raters:
            present = hpo in per_rater_present.get(r, set())
            absent = hpo in per_rater_absent.get(r, set())

            if present:
                rating = 2  # present
            elif absent:
                rating = 1  # absent
            else:
                rating = 0  # unknown (third real category)
                if not include_unknown:
                    continue

            records.append(
                {
                    "image_id": image_id,
                    "hpo": hpo,
                    "rater": r,
                    "rating": rating,
                }
            )

ratings = pd.DataFrame(records)
ratings

,image_id,hpo,rater,rating
0,1,HP:0000293,petkraw2,2
1,2,HP:0000154,petkraw2,2
2,2,HP:0000160,petkraw2,1
3,2,HP:0000179,petkraw2,2
4,2,HP:0000219,petkraw2,1
...,...,...,...,...
66084,27647,HP:0011231,fsantoss,1
66085,27647,HP:0012810,fsantoss,2
66086,27647,HP:0045075,fsantoss,2
66087,27647,HP:0100539,fsantoss,1


In [33]:
# -------------------------------------------------------------------
# Compute Fleiss' kappa per (image, HPO) and per HPO across all images
# -------------------------------------------------------------------
results_image_hpo = []
results_hpo_overall = []

# categories used: 0,1,2
categories = [0, 1, 2] if include_unknown else [1, 2]
cat_to_idx = {c: i for i, c in enumerate(categories)}

# First: Fleiss' kappa per (image_id, hpo)
for (image_id, hpo), group in ratings.groupby(["image_id", "hpo"]):
    # Build counts for this single item: categories × raters
    counts = np.zeros((1, len(categories)), dtype=float)
    for r in group["rating"]:
        counts[0, cat_to_idx[r]] += 1

    kappa = fleiss_kappa(counts)
    results_image_hpo.append(
        {
            "image_id": image_id,
            "hpo": hpo,
            "fleiss_kappa": kappa,
        }
    )

image_hpo_kappa_df = pd.DataFrame(results_image_hpo)
image_hpo_kappa_df.to_csv("fleiss_kappa_per_image_hpo.csv", index=False)

# Second: average Fleiss' kappa per HPO across all images
hpo_avg_results = []
for hpo, hpo_group in image_hpo_kappa_df.groupby("hpo"):
    # Drop NaNs (e.g., HPOs with fewer than 2 ratings on some images)
    valid_kappas = hpo_group["fleiss_kappa"].dropna()
    if valid_kappas.empty:
        avg_kappa = np.nan
        std_kappa = np.nan
        ci_low_kappa = np.nan
        ci_high_kappa = np.nan
    else:
        avg_kappa = valid_kappas.mean()
        std_kappa = valid_kappas.std()
        ci_low_kappa, ci_high_kappa = scipy.stats.t.interval(confidence, len(valid_kappas) - 1, loc=avg_kappa,
                                                             scale=std_kappa)

    hpo_avg_results.append(
        {
            "hpo": hpo,
            "fleiss_kappa_mean": avg_kappa,
            "fleiss_kappa_std": std_kappa,
            "fleiss_kappa_ci_low": ci_low_kappa,
            "fleiss_kappa_ci_high": ci_high_kappa,
            "num_images": len(valid_kappas),
        }
    )

hpo_kappa_mean_df = pd.DataFrame(hpo_avg_results).sort_values(
    "fleiss_kappa_mean", ascending=False
)

hpo_kappa_mean_df.to_csv("fleiss_kappa_mean_per_hpo.csv", index=False)
hpo_kappa_mean_df

C:\Users\hellmafa\Workspace\FaceMeshSyndromes\.venv\Lib\site-packages\scipy\stats\_distn_infrastructure.py:2285: RuntimeWarning: invalid value encountered in multiply
  lower_bound = _a * scale + loc
C:\Users\hellmafa\Workspace\FaceMeshSyndromes\.venv\Lib\site-packages\scipy\stats\_distn_infrastructure.py:2286: RuntimeWarning: invalid value encountered in multiply
  upper_bound = _b * scale + loc


,hpo,fleiss_kappa_mean,fleiss_kappa_std,fleiss_kappa_ci_low,fleiss_kappa_ci_high,num_images
91,HP:0000582,1.0,0.0,NaN,NaN,18
154,HP:0003196,1.0,0.0,NaN,NaN,20
150,HP:0002714,1.0,0.0,NaN,NaN,13
153,HP:0003189,1.0,0.0,NaN,NaN,21
222,HP:0011829,1.0,0.0,NaN,NaN,5
...,...,...,...,...,...,...
249,HP:0100790,NaN,NaN,NaN,NaN,0
250,HP:0100874,NaN,NaN,NaN,NaN,0
251,HP:0100876,NaN,NaN,NaN,NaN,0
252,HP:0400000,NaN,NaN,NaN,NaN,0
